# velo ia — treino LoRA no Google Colab

Este notebook treina o adaptador LoRA da velo ia usando uma GPU T4 gratuita do Colab.

**Antes de rodar:** `Runtime → Change runtime type → T4 GPU`.

O dataset de exemplo tem só 10 diálogos e serve para validar o pipeline. Para uma IA de produção, troque `ai/dataset_amigo.jsonl` por um dataset maior e autorizado.

## 1. Clonar o repositório e instalar as dependências

In [ ]:
!git clone https://github.com/uendersonisrael8-netizen/velo.git /content/velo
%cd /content/velo
!pip install -q --upgrade pip
!pip install -q -r ai/requirements-train.txt

## 2. Conferir a GPU

Se `torch.cuda.is_available()` for `False`, volte em **Runtime → Change runtime type** e escolha **T4 GPU**.

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM (GB):', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))

## 3. Treinar o adaptador

Ajuste `EPOCHS`, `MAX_SEQ_LEN` e o modelo se quiser. O Qwen 2.5 3B Instruct é aberto e não exige aceite de licença. Se usar Llama 3.2, aceite a licença no Hugging Face e faça login com `huggingface_hub.login()`.

In [ ]:
MODEL = 'Qwen/Qwen2.5-3B-Instruct'
EPOCHS = 3
MAX_SEQ_LEN = 1024

!python ai/train_velo_ia.py \
  --model {MODEL} \
  --dataset ai/dataset_amigo.jsonl \
  --output-dir ai/velo-ia-lora \
  --epochs {EPOCHS} \
  --batch-size 1 \
  --grad-accum 8 \
  --lr 2e-4 \
  --max-seq-len {MAX_SEQ_LEN}

## 4. Testar o adaptador

Carrega o modelo em 4-bit e gera uma resposta de exemplo.

In [ ]:
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

quant = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(
    MODEL, quantization_config=quant, device_map='auto'
)
model = PeftModel.from_pretrained(model, 'ai/velo-ia-lora')
model.eval()

messages = [
    {'role': 'system', 'content': 'Você é a velo ia, uma amiga virtual empática e informal.'},
    {'role': 'user', 'content': 'oi, tô ansioso com o trabalho'},
]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
with torch.inference_mode():
    out = model.generate(**inputs, max_new_tokens=180, do_sample=True, temperature=0.8, top_p=0.92)
print(tokenizer.decode(out[0][inputs['input_ids'].shape[-1]:], skip_special_tokens=True))

## 5. Baixar o adaptador

O zip fica em `/content/velo-ia-lora.zip`. Depois, publique no Hub com `hf upload` (veja o `ai/README.md`) e aponte `VELO_IA_ADAPTER` no Space para o repositório.

In [ ]:
import shutil
from google.colab import files

shutil.make_archive('/content/velo-ia-lora', 'zip', 'ai/velo-ia-lora')
files.download('/content/velo-ia-lora.zip')